In [2]:
import pandas as pd
import numpy as np
import os
import glob

DATA_DIR = "/home/arshad/Network-project/data/"
all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

dfs = []
for file in all_files:
    df_temp = pd.read_csv(file, encoding='utf-8', low_memory=False)
    df_temp['source_file'] = os.path.basename(file)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Loaded: 2,830,743 rows × 80 columns


In [3]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"🗑️  Dropped {before - after:,} duplicate rows")
print(f"✅ Remaining: {after:,} rows")

🗑️  Dropped 256,479 duplicate rows
✅ Remaining: 2,574,264 rows


In [4]:
# Replace inf/-inf with NaN first
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f"✅ Replaced infinite values with NaN")

# Check what we now have as NaN
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"\n⚠️  Columns now with NaN:")
print(missing)

✅ Replaced infinite values with NaN

⚠️  Columns now with NaN:
Flow Bytes/s      1624
Flow Packets/s    1624
dtype: int64


In [5]:
# For numeric columns, fill NaN with median (robust to outliers)
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Verify
remaining_missing = df.isnull().sum().sum()
print(f"✅ Missing values remaining: {remaining_missing}")

✅ Missing values remaining: 0


In [6]:
# Fix encoding issues in label names (the ? characters you saw)
df['Label'] = df['Label'].str.strip()
df['Label'] = df['Label'].replace({
    'Web Attack \x96 Brute Force': 'Web Attack - Brute Force',
    'Web Attack \x96 XSS':         'Web Attack - XSS',
    'Web Attack \x96 Sql Injection':'Web Attack - SQL Injection'
})

print("🏷️  Cleaned Label Distribution:")
print(df['Label'].value_counts())

🏷️  Cleaned Label Distribution:
Label
BENIGN                        2148386
DoS Hulk                       172849
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [7]:
# Define our mapping
mitre_mapping = {
    'BENIGN':                       ('None',         'None',   'None'),
    'DoS Hulk':                     ('TA0040',       'T1499',  'Endpoint Denial of Service'),
    'DoS GoldenEye':                ('TA0040',       'T1499',  'Endpoint Denial of Service'),
    'DoS slowloris':                ('TA0040',       'T1499',  'Endpoint Denial of Service'),
    'DoS Slowhttptest':             ('TA0040',       'T1499',  'Endpoint Denial of Service'),
    'DDoS':                         ('TA0040',       'T1498',  'Network Denial of Service'),
    'PortScan':                     ('TA0007',       'T1046',  'Network Service Discovery'),
    'FTP-Patator':                  ('TA0006',       'T1110',  'Brute Force'),
    'SSH-Patator':                  ('TA0006',       'T1110',  'Brute Force'),
    'Bot':                          ('TA0011',       'T1071',  'Application Layer Protocol'),
    'Web Attack - Brute Force':     ('TA0001',       'T1190',  'Exploit Public-Facing Application'),
    'Web Attack - XSS':             ('TA0001',       'T1190',  'Exploit Public-Facing Application'),
    'Web Attack - SQL Injection':   ('TA0001',       'T1190',  'Exploit Public-Facing Application'),
    'Heartbleed':                   ('TA0001',       'T1210',  'Exploitation of Remote Services'),
    'Infiltration':                 ('TA0008',       'T1570',  'Lateral Tool Transfer'),
}

# Apply mapping
df['MITRE_Tactic']     = df['Label'].map(lambda x: mitre_mapping.get(x, ('Unknown','Unknown','Unknown'))[0])
df['MITRE_Technique']  = df['Label'].map(lambda x: mitre_mapping.get(x, ('Unknown','Unknown','Unknown'))[1])
df['MITRE_Tech_Name']  = df['Label'].map(lambda x: mitre_mapping.get(x, ('Unknown','Unknown','Unknown'))[2])

print("✅ MITRE ATT&CK columns added!")
print(df[['Label', 'MITRE_Tactic', 'MITRE_Technique', 'MITRE_Tech_Name']].drop_duplicates())

✅ MITRE ATT&CK columns added!
                              Label MITRE_Tactic MITRE_Technique  \
0                            BENIGN         None            None   
12637      Web Attack � Brute Force      Unknown         Unknown   
72134              Web Attack � XSS      Unknown         Unknown   
90293    Web Attack � Sql Injection      Unknown         Unknown   
176924                DoS slowloris       TA0040           T1499   
239641             DoS Slowhttptest       TA0040           T1499   
245226                     DoS Hulk       TA0040           T1499   
501399                DoS GoldenEye       TA0040           T1499   
767496                   Heartbleed       TA0001           T1210   
881952                         DDoS       TA0040           T1498   
1100161                 FTP-Patator       TA0006           T1110   
1250803                 SSH-Patator       TA0006           T1110   
1536186                    PortScan       TA0007           T1046   
2375180           

In [8]:
OUTPUT_PATH = "/home/arshad/Network-project/data/cleaned_cicids2017.csv"
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Cleaned dataset saved!")
print(f"📊 Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📁 Location: {OUTPUT_PATH}")


✅ Cleaned dataset saved!
📊 Final shape: 2,574,264 rows × 83 columns
📁 Location: /home/arshad/Network-project/data/cleaned_cicids2017.csv


In [9]:
# Check the EXACT characters in the problematic labels
print("Exact bytes in Web Attack labels:")
for label in df['Label'].unique():
    if 'Web' in str(label):
        print(f"  repr: {repr(label)}")

Exact bytes in Web Attack labels:
  repr: 'Web Attack � Brute Force'
  repr: 'Web Attack � XSS'
  repr: 'Web Attack � Sql Injection'


In [10]:
# The ? is \x96 in Windows-1252 encoding (en-dash character)
df['Label'] = df['Label'].str.encode('latin-1', errors='replace').str.decode('latin-1')

df['Label'] = df['Label'].str.replace(r'Web Attack\s+.\s+Brute Force', 'Web Attack - Brute Force', regex=True)
df['Label'] = df['Label'].str.replace(r'Web Attack\s+.\s+XSS', 'Web Attack - XSS', regex=True)
df['Label'] = df['Label'].str.replace(r'Web Attack\s+.\s+Sql Injection', 'Web Attack - SQL Injection', regex=True)

# Verify
print("✅ Fixed labels:")
for label in df['Label'].unique():
    if 'Web' in str(label):
        print(f"  → {label}")

✅ Fixed labels:
  → Web Attack - Brute Force
  → Web Attack - XSS
  → Web Attack - SQL Injection


In [11]:
mitre_mapping = {
    'BENIGN':                        ('None',   'None',  'None'),
    'DoS Hulk':                      ('TA0040', 'T1499', 'Endpoint Denial of Service'),
    'DoS GoldenEye':                 ('TA0040', 'T1499', 'Endpoint Denial of Service'),
    'DoS slowloris':                 ('TA0040', 'T1499', 'Endpoint Denial of Service'),
    'DoS Slowhttptest':              ('TA0040', 'T1499', 'Endpoint Denial of Service'),
    'DDoS':                          ('TA0040', 'T1498', 'Network Denial of Service'),
    'PortScan':                      ('TA0007', 'T1046', 'Network Service Discovery'),
    'FTP-Patator':                   ('TA0006', 'T1110', 'Brute Force'),
    'SSH-Patator':                   ('TA0006', 'T1110', 'Brute Force'),
    'Bot':                           ('TA0011', 'T1071', 'Application Layer Protocol'),
    'Web Attack - Brute Force':      ('TA0001', 'T1190', 'Exploit Public-Facing Application'),
    'Web Attack - XSS':              ('TA0001', 'T1190', 'Exploit Public-Facing Application'),
    'Web Attack - SQL Injection':    ('TA0001', 'T1190', 'Exploit Public-Facing Application'),
    'Heartbleed':                    ('TA0001', 'T1210', 'Exploitation of Remote Services'),
    'Infiltration':                  ('TA0008', 'T1570', 'Lateral Tool Transfer'),
}

df['MITRE_Tactic']    = df['Label'].map(lambda x: mitre_mapping.get(x, ('Unknown','Unknown','Unknown'))[0])
df['MITRE_Technique'] = df['Label'].map(lambda x: mitre_mapping.get(x, ('Unknown','Unknown','Unknown'))[1])
df['MITRE_Tech_Name'] = df['Label'].map(lambda x: mitre_mapping.get(x, ('Unknown','Unknown','Unknown'))[2])

print("✅ MITRE mapping re-applied!")
print(df[['Label','MITRE_Tactic','MITRE_Technique','MITRE_Tech_Name']].drop_duplicates().to_string())

✅ MITRE mapping re-applied!
                              Label MITRE_Tactic MITRE_Technique                    MITRE_Tech_Name
0                            BENIGN         None            None                               None
12637      Web Attack - Brute Force       TA0001           T1190  Exploit Public-Facing Application
72134              Web Attack - XSS       TA0001           T1190  Exploit Public-Facing Application
90293    Web Attack - SQL Injection       TA0001           T1190  Exploit Public-Facing Application
176924                DoS slowloris       TA0040           T1499         Endpoint Denial of Service
239641             DoS Slowhttptest       TA0040           T1499         Endpoint Denial of Service
245226                     DoS Hulk       TA0040           T1499         Endpoint Denial of Service
501399                DoS GoldenEye       TA0040           T1499         Endpoint Denial of Service
767496                   Heartbleed       TA0001           T1210    Expl

In [12]:
# Check for any remaining unknowns
unknown_mask = df['MITRE_Technique'] == 'Unknown'
unknown_count = unknown_mask.sum()

if unknown_count == 0:
    print("✅ No Unknown mappings remaining!")
else:
    print(f"⚠️  Still {unknown_count:,} Unknown rows:")
    print(df[unknown_mask]['Label'].value_counts())

# Re-save the fixed dataset
OUTPUT_PATH = "/home/arshad/Network-project/data/cleaned_cicids2017.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Fixed dataset re-saved!")
print(f"📊 Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ No Unknown mappings remaining!

✅ Fixed dataset re-saved!
📊 Final shape: 2,574,264 rows × 83 columns
